In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

openai_client = OpenAI(
    api_key = os.getenv("GROQ_API_KEY"),
    base_url = "https://api.groq.com/openai/v1"
)



In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data , build_index

documents = load_faq_data()
index = build_index(documents)


In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index = index,
    llm_client = openai_client,
    instructions = instructions,
)

In [4]:
assistant.rag("How do I run Ollama locally")

'To run Ollama locally, follow these steps:\n\n1. Install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n   - **macOS**: Download the `.pkg` and install it.\n   - **Windows**: Download the `.msi` and install it.\n   - **Linux**: Run the command `curl -fsSL https://ollama.com/install.sh | sh` in the terminal.\n\n2. Open a terminal and type `ollama run llama3` to download the LLaMA 3 model (~4GB) and start it locally. This will also open a chat-like interface where you can type questions.\n\n3. To test the Ollama local server, run `curl http://localhost:11434` and verify you receive a response similar to `{"models": [...]}`.\n\n4. Install the Python client with `pip install ollama`.\n\nYou can then use the Ollama Python client to interact with the local Ollama server. A minimal example is provided:\n```python\nimport ollama\n\nresponse = ollama.chat(\n    model=\'llama3\',\n    messages=[{"role": "user", "content": your_

In [5]:
assistant.rag("How do I run Olama locally?")

"There is no information in the provided CONTEXT about how to run Olama locally. The CONTEXT only mentions running the course locally in general, without specifying how to run Olama. Therefore, I couldn't find a relevant answer to your question."

In [6]:
messages = [
    {"role": "user", "content": "How do I run Ollama locally?"}
    ]
response = openai_client.chat.completions.create(
    model = "llama-3.3-70b-versatile",
    messages = messages,
)

response.choices[0].message.content

"To run Ollama locally, you'll need to follow these steps. Please note that these instructions are subject to change as the Ollama project evolves.\n\n### Prerequisites\n1. **Python**: Ensure you have Python 3.9 or later installed on your machine. You can download it from the official Python website if needed.\n2. **pip**: Make sure you have pip, the package installer for Python, installed. It usually comes with Python.\n3. **Virtual Environment**: While not strictly necessary, using a virtual environment (like `venv`) is recommended to keep your project dependencies isolated.\n4. **Git**: You'll need Git to clone the Ollama repository. Download and install it from the Git website if you haven't already.\n\n### Setup Instructions\n1. **Clone the Ollama Repository**:\n   Open your terminal or command prompt and run:\n   ```bash\n   git clone https://github.com/ollama-dev/ollama.git\n   ```\n   This command clones the Ollama repository to a folder named `ollama` in your current directory

In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query, 
        num_results = 5,
        boost_dict = boost_dict,
        filter_dict = filter_dict 
    )



In [8]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            # "additionalProperties": False
        }
    }
}

In [9]:
response = openai_client.chat.completions.create(
    model="qwen/qwen3.6-27b",
    messages=messages,
    tools=[search_tool],
)

response.choices[0].message.content

In [10]:
import json

tool_call = response.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)

results = search(**args)
result_json = json.dumps(results, indent = 2)

In [13]:
messages.extend(response.choices[0].message)

messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "conetent": result_json,
})


In [14]:
response = openai_client.chat.completions.create(
    model="qwen/qwen3.6-27b",
    messages=messages,
    tools=[search_tool],
)

print(response.choices[0].message.content)

BadRequestError: Error code: 400 - {'error': {'message': "'messages.1' : value must be an object with the discriminator property: 'role'", 'type': 'invalid_request_error'}}